# CNN Spatial Feature Extraction
**Google Colab notebook**  
This notebook trains a compact CNN on MNIST and visualizes convolution filters, feature maps, pooling, and global average pooling.

**Colab setup:** Runtime → Change runtime type → T4 GPU (optional). CPU is also supported. Run top to bottom.

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Load and inspect MNIST
MNIST contains 28×28 grayscale handwritten digit images with labels 0–9.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixels to [0, 1] and add the channel dimension.
x_train = (x_train.astype("float32") / 255.0)[..., np.newaxis]
x_test = (x_test.astype("float32") / 255.0)[..., np.newaxis]

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.flat, x_train[:10], y_train[:10]):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Convolution output-size calculator

For each spatial dimension:

\[
\text{output} = \left\lfloor\frac{N+2P-K}{S}\right\rfloor+1
\]

Try changing the values below. For a 28×28 input, 3×3 kernel, stride 1, and valid padding, the output is 26×26.

In [ ]:
def conv_output_size(n, kernel, stride=1, padding=0):
    return (n + 2 * padding - kernel) // stride + 1

print("28x28, 3x3, stride 1, valid:",
      conv_output_size(28, 3), "x", conv_output_size(28, 3))
print("28x28, 3x3, stride 2, valid:",
      conv_output_size(28, 3, stride=2), "x",
      conv_output_size(28, 3, stride=2))

## 4. A tiny hand-crafted convolution example

This demonstrates the multiply-and-sum operation. The example is a simple 2×2 filter applied at the top-left of a 3×3 patch. Deep-learning libraries learn filter weights during training.

In [ ]:
patch = np.array([[1, 2, 0],
                   [0, 1, 3],
                   [2, 1, 0]], dtype=np.float32)
kernel = np.array([[1, 0],
                   [0, -1]], dtype=np.float32)

top_left = np.sum(patch[:2, :2] * kernel)
print("Top-left convolution result:", top_left)

# Show all valid positions for this small example.
feature = np.zeros((2, 2), dtype=np.float32)
for i in range(2):
    for j in range(2):
        feature[i, j] = np.sum(patch[i:i+2, j:j+2] * kernel)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(patch, cmap="gray")
axes[0].set_title("Input patch")
axes[1].imshow(kernel, cmap="coolwarm")
axes[1].set_title("2×2 filter")
axes[2].imshow(feature, cmap="viridis")
axes[2].set_title("Output feature map")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Build a compact CNN

Architecture:
- Conv2D: 16 filters, 3×3, ReLU
- MaxPooling2D: 2×2
- Conv2D: 32 filters, 3×3, ReLU
- MaxPooling2D: 2×2
- Flatten and Dense classifier

The model learns its filters from labelled examples.

In [ ]:
inputs = keras.Input(shape=(28, 28, 1), name="image")
x = layers.Conv2D(16, 3, activation="relu", name="conv1")(inputs)
x = layers.MaxPooling2D(2, name="pool1")(x)
x = layers.Conv2D(32, 3, activation="relu", name="conv2")(x)
x = layers.MaxPooling2D(2, name="pool2")(x)
x = layers.Flatten(name="flatten")(x)
x = layers.Dense(64, activation="relu", name="dense_features")(x)
outputs = layers.Dense(10, activation="softmax", name="predictions")(x)

model = keras.Model(inputs, outputs, name="mnist_cnn")
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

## 6. Train the CNN
For a classroom demo, 3–5 epochs is usually enough to see learning. Increase epochs if time and runtime allow.

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=4,
    batch_size=128,
    verbose=1
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.title("Accuracy")
plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("Loss")
plt.tight_layout()
plt.show()

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

## 7. Visualize learned first-layer filters

Each first-layer filter spans the one grayscale input channel and produces one output feature map. The weights are learned during training. They may look like edge/contrast detectors, but do not assume every filter has a simple human interpretation.

In [ ]:
conv1 = model.get_layer("conv1")
weights, biases = conv1.get_weights()
print("Filter tensor shape:", weights.shape)  # (3, 3, 1, 16)

n_filters = weights.shape[-1]
fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(weights[:, :, 0, i], cmap="coolwarm")
    ax.set_title(f"Filter {i}")
    ax.axis("off")
plt.suptitle("Learned Conv1 filter weights")
plt.tight_layout()
plt.show()

## 8. Visualize feature maps for one image

A feature map shows where a filter responds to this particular input. Different channels respond differently.

In [ ]:
sample = x_test[0:1]
sample_label = int(y_test[0])

activation_model = keras.Model(
    inputs=model.input,
    outputs=[model.get_layer("conv1").output,
             model.get_layer("pool1").output,
             model.get_layer("conv2").output]
)
conv1_act, pool1_act, conv2_act = activation_model.predict(sample, verbose=0)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(sample[0, :, :, 0], cmap="gray")
axes[0].set_title(f"Input (label {sample_label})")
for i in range(3):
    axes[0].axis("off")
for ax, fmap, title in zip(axes[1:], [conv1_act, pool1_act, conv2_act],
                           ["Conv1 channel 0", "Pool1 channel 0", "Conv2 channel 0"]):
    ax.imshow(fmap[0, :, :, 0], cmap="viridis")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Conv1 activation shape:", conv1_act.shape)
print("Pool1 activation shape:", pool1_act.shape)
print("Conv2 activation shape:", conv2_act.shape)

### Inspect multiple channels

In [ ]:
n_show = 8
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(conv1_act[0, :, :, i], cmap="viridis")
    ax.set_title(f"Channel {i}")
    ax.axis("off")
plt.suptitle("First convolution layer: feature maps")
plt.tight_layout()
plt.show()

## 9. Compare max pooling and average pooling

The next cell applies both pooling operations to the same activation tensor. Max pooling retains the strongest local response; average pooling summarizes the local average.

In [ ]:
activation_tensor = tf.constant(conv1_act)
max_pooled = layers.MaxPooling2D(pool_size=2, strides=2)(activation_tensor)
avg_pooled = layers.AveragePooling2D(pool_size=2, strides=2)(activation_tensor)

channel = 0
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(conv1_act[0, :, :, channel], cmap="viridis")
axes[0].set_title("Original feature map")
axes[1].imshow(max_pooled.numpy()[0, :, :, channel], cmap="viridis")
axes[1].set_title("Max pooled")
axes[2].imshow(avg_pooled.numpy()[0, :, :, channel], cmap="viridis")
axes[2].set_title("Average pooled")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

print("Before pooling:", conv1_act.shape)
print("After max pooling:", max_pooled.shape)
print("After average pooling:", avg_pooled.shape)

## 10. Global average pooling (GAP)

GAP averages each feature channel across all spatial positions. A tensor with shape `7 × 7 × 32` becomes a vector with 32 values for each example.

In [ ]:
gap_layer = layers.GlobalAveragePooling2D()
gap_values = gap_layer(conv2_act).numpy()

print("Conv2 tensor shape:", conv2_act.shape)
print("After GAP:", gap_values.shape)

plt.figure(figsize=(10, 3))
plt.bar(np.arange(gap_values.shape[-1]), gap_values[0])
plt.xlabel("Feature channel")
plt.ylabel("Average activation")
plt.title("One GAP value per feature channel")
plt.show()

### Compare classifier heads: Flatten vs GAP

This is an architecture illustration using the same feature-map dimensions. GAP usually creates a much smaller vector than Flatten. Parameter counts depend on the following dense layer.

In [ ]:
feature_shape = model.get_layer("pool2").output.shape
print("Feature map before classifier:", feature_shape)

# Build two small illustrative heads from a 7x7x32 tensor.
flatten_input = keras.Input(shape=(7, 7, 32))
flat_output = layers.Dense(64, activation="relu")(layers.Flatten()(flatten_input))
flatten_head = keras.Model(flatten_input, flat_output)

gap_input = keras.Input(shape=(7, 7, 32))
gap_output = layers.Dense(64, activation="relu")(
    layers.GlobalAveragePooling2D()(gap_input)
)
gap_head = keras.Model(gap_input, gap_output)

print("Flatten head parameters:", flatten_head.count_params())
print("GAP head parameters:", gap_head.count_params())

## 11. Make a prediction and inspect confidence

Softmax values are class scores that sum to approximately 1. They are not guaranteed to be perfectly calibrated probabilities.

In [ ]:
predictions = model.predict(x_test[:8], verbose=0)
predicted_labels = predictions.argmax(axis=1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i].squeeze(), cmap="gray")
    ax.set_title(f"True: {y_test[i]} | Pred: {predicted_labels[i]}\nScore: {predictions[i, predicted_labels[i]]:.2f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 12. Student investigation tasks

Complete these after the instructor walkthrough:

1. Change Conv1 from 16 filters to 32. What changes in the activation tensor shape?
2. Replace `MaxPooling2D` with `AveragePooling2D`. Compare test accuracy and visual responses.
3. Change the kernel size from 3×3 to 5×5. What happens to the feature-map dimensions with `padding="valid"`?
4. Replace `Flatten()` with `GlobalAveragePooling2D()`. Compare model parameter count and accuracy.
5. Choose one correctly classified and one incorrectly classified image. Visualize their feature maps and suggest why the second image may be difficult.

**Record your results:** model change, parameter count, test accuracy, and one visualization observation.

## 13. Instructor wrap-up questions

- What does a convolution filter learn?
- How is a feature map different from the filter itself?
- Why does parameter sharing help?
- What information can pooling remove?
- What does GAP do to spatial dimensions?
- How does a ResNet skip connection help deep networks?
- Why might an Inception module use multiple filter sizes in parallel?

**Reminder:** LeNet, AlexNet, VGG, ResNet, and Inception are discussed in the accompanying `CNN_Training_Notes.md`. This notebook focuses on the core spatial operations rather than reproducing those historical architectures.